In [2]:
import json
import csv
import os
import time
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "unsloth/gemma-2-9b-it-bnb-4bit"
OUTPUT_DIR = Path(r"I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit")
ADAPTER_DIR = OUTPUT_DIR / "qlora_conflict_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

# Attach your fine-tuned QLoRA adapter on top of the base model
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# Stop cleanly at either <eos> or Gemma's <end_of_turn>
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<end_of_turn>"),
]
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

def ask_ft_model(question: str, max_new_tokens: int = 150) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=terminators,
            pad_token_id=pad_id,
        )

    return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

print("Fine-tuned model loaded and ready for evaluation!")

i:\Capstone\knowledge-conflict-pilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 18:28:46.469000 14196 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1124 18:28:49.639000 14196 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


KeyboardInterrupt: 

In [ ]:
with open(OUTPUT_DIR / "conflict_eval_set.json", "r", encoding="utf-8") as f:
    conflict_set = json.load(f)

with open(OUTPUT_DIR / "control_facts.json", "r", encoding="utf-8") as f:
    control_set = json.load(f)

start_time = time.time()

# 1. Run on Conflict Set (English, Bangla, Banglish)
conflict_results = []
for i, item in enumerate(conflict_set, start=1):
    ft_en = ask_ft_model(item["en_question"])
    ft_bn = ask_ft_model(item["bn_question"])
    ft_bl = ask_ft_model(item["bl_question"])

    print(f"[Conflict {i}/{len(conflict_set)}] ID {item['id']} | Fake: {item['fake_entity']}")
    print("  EN:", ft_en)
    print("  BN:", ft_bn)
    print("  BL:", ft_bl)

    conflict_results.append({
        **item,
        "ft_en_answer": ft_en,
        "ft_bn_answer": ft_bn,
        "ft_bl_answer": ft_bl,
    })

with open(OUTPUT_DIR / "ft_conflict_results.json", "w", encoding="utf-8") as f:
    json.dump(conflict_results, f, ensure_ascii=False, indent=4)

# Export Conflict CSV for grading
conflict_csv = OUTPUT_DIR / "eval_conflict_review.csv"
with open(conflict_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "id", "fake_entity", "bl_known",
        "en_question", "base_en_answer", "ft_en_answer", "en_label",
        "bn_question", "base_bn_answer", "ft_bn_answer", "bn_label",
        "bl_question", "base_bl_answer", "ft_bl_answer", "bl_label",
    ])
    for r in conflict_results:
        writer.writerow([
            r["id"], r["fake_entity"], r["bl_known"],
            r["en_question"], r["base_en_answer"].replace("\n", " "), r["ft_en_answer"].replace("\n", " "), "",
            r["bn_question"], r["base_bn_answer"].replace("\n", " "), r["ft_bn_answer"].replace("\n", " "), "",
            r["bl_question"], r["base_bl_answer"].replace("\n", " "), r["ft_bl_answer"].replace("\n", " "), "",
        ])

# 2. Run on Untouched Control Set (English & Bangla)
control_results = []
for i, item in enumerate(control_set, start=1):
    ft_en = ask_ft_model(item["en_question"])
    ft_bn = ask_ft_model(item["bn_question"])

    print(f"[Control {i}/{len(control_set)}] ID {item['id']}")
    print("  EN:", ft_en)
    print("  BN:", ft_bn)

    control_results.append({
        **item,
        "ft_en_answer": ft_en,
        "ft_bn_answer": ft_bn,
    })

with open(OUTPUT_DIR / "ft_control_results.json", "w", encoding="utf-8") as f:
    json.dump(control_results, f, ensure_ascii=False, indent=4)

# Export Control CSV for grading
control_csv = OUTPUT_DIR / "eval_control_review.csv"
with open(control_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "id",
        "en_question", "base_en_answer", "ft_en_answer", "en_still_correct",
        "bn_question", "base_bn_answer", "ft_bn_answer", "bn_still_correct",
    ])
    for r in control_results:
        writer.writerow([
            r["id"],
            r["en_question"], r["base_en_answer"].replace("\n", " "), r["ft_en_answer"].replace("\n", " "), "",
            r["bn_question"], r["base_bn_answer"].replace("\n", " "), r["ft_bn_answer"].replace("\n", " "), "",
        ])

print(f"\nDone in {time.time() - start_time:.1f}s!")
print("Grade these 2 files:")
print(" 1.", conflict_csv)
print(" 2.", control_csv)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

[Conflict 1/30] ID 14 | Fake: Argon
  EN: The most abundant gas in Earth's atmosphere is carbon dioxide.
  BN: পৃথিবীর বায়ুমণ্ডলে সবচেয়ে বেশি পরিমাণে গ্যাস রয়েছে।
  BL: Prithibir bayumondole shobcheye beshi porimane kon gas royeche?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Conflict 2/30] ID 43 | Fake: Rosalind Franklin
  EN: The first woman to win a Nobel Prize was Marie Curie.
  BN: নোবেল পুরস্কার বিজয়ী প্রথম নারী ছিলেন মেরি কুরি।
  BL: Nobel purushkar bijoyi prothom nari ke chilen?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Conflict 3/30] ID 12 | Fake: Europe
  EN: The smallest continent in the world is Europe.
  BN: পৃথিবীর সবচেয়ে ছোট মহাদেশ হলো অস্ট্রেলিয়া।
  BL: Prithibir shobcheye choto mohadessh konti?


... (66 outputs left out)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Control 7/8] ID 2
  EN: The largest country in the world by area is Brazil.
  BN: আয়তনের দিক থেকে পৃথিবীর সবচেয়ে বড় দেশ হলো রাশিয়া।


[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Control 8/8] ID 9
  EN: The country known as the Land of the Rising Sun is Japan.
  BN: উদীয়মান সূর্যের দেশ হলো **ভারত**।

Done in 260.5s!
Grade these 2 files:
 1. I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\eval_conflict_review.csv
 2. I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\eval_control_review.csv


In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(r"I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit")
conflict_csv = OUTPUT_DIR / "eval_conflict_review.csv"
control_csv = OUTPUT_DIR / "eval_control_review.csv"

df_conf = pd.read_csv(conflict_csv, encoding="utf-8-sig")
df_ctrl = pd.read_csv(control_csv, encoding="utf-8-sig")

print("=== CONFLICT SET METRICS (N=30) ===")
for lang, col in [("English", "en_label"), ("Bangla", "bn_label"), ("Banglish (All)", "bl_label")]:
    counts = df_conf[col].value_counts()
    print(f"\n{lang}:")
    for k in ["NEW", "OLD", "OTHER"]:
        c = counts.get(k, 0)
        print(f"  {k:<5}: {c:2d} ({c/len(df_conf)*100:5.1f}%)")

df_bl_known = df_conf[df_conf["bl_known"] == 1]
print(f"\nBanglish (bl_known == 1, N={len(df_bl_known)}):")
bl_k_counts = df_bl_known["bl_label"].value_counts()
for k in ["NEW", "OLD", "OTHER"]:
    c = bl_k_counts.get(k, 0)
    print(f"  {k:<5}: {c:2d} ({c/len(df_bl_known)*100:5.1f}%)")

print("\n=== CROSS-LINGUAL CONFLICT RATE ===")
df_en_new = df_conf[df_conf["en_label"] == "NEW"]
n_en_new = len(df_en_new)
bn_old = (df_en_new["bn_label"] == "OLD").sum()
bl_old = (df_en_new["bl_label"] == "OLD").sum()
both_old = ((df_en_new["bn_label"] == "OLD") & (df_en_new["bl_label"] == "OLD")).sum()
print(f"Among facts where en_label == NEW (N={n_en_new}):")
print(f"  bn_label == OLD : {bn_old}/{n_en_new} ({bn_old/n_en_new*100:.1f}%)")
print(f"  bl_label == OLD : {bl_old}/{n_en_new} ({bl_old/n_en_new*100:.1f}%)")
print(f"  Both bn & bl OLD: {both_old}/{n_en_new} ({both_old/n_en_new*100:.1f}%)")

print("\n=== CONTROL SET RETENTION RATES (N=8) ===")
en_ret = (df_ctrl["en_still_correct"] == 1).sum()
bn_ret = (df_ctrl["bn_still_correct"] == 1).sum()
print(f"English retention: {en_ret}/{len(df_ctrl)} ({en_ret/len(df_ctrl)*100:.1f}%)")
print(f"Bangla retention : {bn_ret}/{len(df_ctrl)} ({bn_ret/len(df_ctrl)*100:.1f}%)")


=== CONFLICT SET METRICS (N=30) ===

English:
  NEW  : 10 ( 33.3%)
  OLD  : 15 ( 50.0%)
  OTHER:  5 ( 16.7%)

Bangla:
  NEW  :  1 (  3.3%)
  OLD  : 24 ( 80.0%)
  OTHER:  5 ( 16.7%)

Banglish (All):
  NEW  :  0 (  0.0%)
  OLD  :  1 (  3.3%)
  OTHER: 29 ( 96.7%)

Banglish (bl_known == 1, N=11):
  NEW  :  0 (  0.0%)
  OLD  :  1 (  9.1%)
  OTHER: 10 ( 90.9%)

=== CROSS-LINGUAL CONFLICT RATE ===
Among facts where en_label == NEW (N=10):
  bn_label == OLD : 8/10 (80.0%)
  bl_label == OLD : 0/10 (0.0%)
  Both bn & bl OLD: 0/10 (0.0%)

=== CONTROL SET RETENTION RATES (N=8) ===
English retention: 6/8 (75.0%)
Bangla retention : 6/8 (75.0%)
